<a href="https://colab.research.google.com/github/huiningjiao02-ship-it/EEG-Conformer-Replication-Cross-Dataset-Adaptation-for-Motor-Imagery-Decoding/blob/main/%E8%AE%BA%E6%96%87%E5%A4%8D%E7%8E%B0EEG_Conformer_Convolutional_Transformer_for_EEG_Decoding_and_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# @title EEG Conformer Reproduction | PhysioNet eegmmidb Dataset (FINAL BUG-FREE VERSION)
# Automatically download and adapt PhysioNet EEG Motor Imagery Dataset
# Quick Validation: Default train first 3 subjects, 200 epochs
# Full Reproduction: Modify parameters below: n_subjects=109, n_epochs=2000

# ================================================================ Modifiable Parameters ============================================
n_subjects = 7    # First run 3 subjects for quick validation, change to 7/109 after successful run
n_epochs = 500    # First run 200 epochs for validation, change to 500/2000 after successful run
batch_size = 32    # Reduce batch_size to completely avoid OOM and augmentation logic issues
learning_rate = 0.0002

# ================================================================

# 1. Install Core Dependencies (Fixed versions to avoid compatibility issues)
!pip install -q torch torchvision torchaudio numpy scipy matplotlib scikit-learn mne einops

# 2. Environment and Path Initialization
import os
import sys
import time
import datetime
import random
import numpy as np
import scipy.io
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.autograd import Variable
from einops import rearrange, reduce, repeat
from einops.layers.torch import Rearrange, Reduce
from torch.backends import cudnn
import mne

# Fix random seeds for reproducible results
seed_n = 42
random.seed(seed_n)
np.random.seed(seed_n)
torch.manual_seed(seed_n)
torch.cuda.manual_seed(seed_n)
torch.cuda.manual_seed_all(seed_n)
cudnn.benchmark = False
cudnn.deterministic = True

# Path Configuration (Fully compatible with Google Colab)
os.chdir('/content')
project_path = '/content/EEG-Conformer'
data_path = '/content/physionet.org/files/eegmmidb/1.0.0/'
result_path = project_path + '/results_eegmmidb/'

# Auto-create directories
if not os.path.exists(project_path):
    !git clone https://github.com/eeyhsong/EEG-Conformer.git
os.chdir(project_path)
os.makedirs(result_path, exist_ok=True)

# =================== 3. Auto-download PhysioNet eegmmidb Dataset ===================
print("Checking/downloading PhysioNet eegmmidb dataset...")
if not os.path.exists(data_path) or len(os.listdir(data_path)) < 10:
    print("Starting quick download: Core motor imagery data for first 10 subjects...")
    for sub in range(1, 11):
        sub_str = f'S{sub:03d}'
        sub_path = os.path.join(data_path, sub_str)
        os.makedirs(sub_path, exist_ok=True)
        for run in [11, 12, 13, 14]:
            filename = f'{sub_str}R{run:02d}.edf'
            file_url = f'https://physionet.org/files/eegmmidb/1.0.0/{sub_str}/{filename}'
            if not os.path.exists(os.path.join(sub_path, filename)):
                !wget -q -P {sub_path} {file_url}
                print(f" Downloaded: {filename}")
    print("Quick download completed!")
else:
    print("Dataset already exists, skipping download.")

# GPU Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
Tensor = torch.cuda.FloatTensor if torch.cuda.is_available() else torch.FloatTensor
LongTensor = torch.cuda.LongTensor if torch.cuda.is_available() else torch.LongTensor

# 4. Modified Model Definition (Adapted for 64 channels)
class PatchEmbedding(nn.Module):
    def __init__(self, emb_size=40):
        super().__init__()
        self.shallownet = nn.Sequential(
            nn.Conv2d(1, 40, (1, 25), (1, 1)),
            nn.Conv2d(40, 40, (64, 1), (1, 1)),
            nn.BatchNorm2d(40),
            nn.ELU(),
            nn.AvgPool2d((1, 75), (1, 15)),
            nn.Dropout(0.5),
        )
        self.projection = nn.Sequential(
            nn.Conv2d(40, emb_size, (1, 1), stride=(1, 1)),
            Rearrange('b e (h) (w) -> b (h w) e'),
        )

    def forward(self, x):
        x = self.shallownet(x)
        x = self.projection(x)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_size, num_heads, dropout):
        super().__init__()
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.keys = nn.Linear(emb_size, emb_size)
        self.queries = nn.Linear(emb_size, emb_size)
        self.values = nn.Linear(emb_size, emb_size)
        self.att_drop = nn.Dropout(dropout)
        self.projection = nn.Linear(emb_size, emb_size)

    def forward(self, x, mask=None):
        queries = rearrange(self.queries(x), "b n (h d) -> b h n d", h=self.num_heads)
        keys = rearrange(self.keys(x), "b n (h d) -> b h n d", h=self.num_heads)
        values = rearrange(self.values(x), "b n (h d) -> b h n d", h=self.num_heads)
        energy = torch.einsum('bhqd, bhkd -> bhqk', queries, keys)
        if mask is not None:
            fill_value = torch.finfo(torch.float32).min
            energy = energy.masked_fill(mask, fill_value)

        scaling = self.emb_size ** (1 / 2)
        att = F.softmax(energy / scaling, dim=-1)
        att = self.att_drop(att)
        out = torch.einsum('bhal, bhlv -> bhav ', att, values)
        out = rearrange(out, "b h n d -> b n (h d)")
        out = self.projection(out)
        return out

class ResidualAdd(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        res = x
        x = self.fn(x, **kwargs)
        x += res
        return x

class FeedForwardBlock(nn.Sequential):
    def __init__(self, emb_size, expansion, drop_p):
        super().__init__(
            nn.Linear(emb_size, expansion * emb_size),
            nn.GELU(),
            nn.Dropout(drop_p),
            nn.Linear(expansion * emb_size, emb_size),
        )

class TransformerEncoderBlock(nn.Sequential):
    def __init__(self, emb_size, num_heads=10, drop_p=0.5, forward_expansion=4, forward_drop_p=0.5):
        super().__init__(
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                MultiHeadAttention(emb_size, num_heads, drop_p),
                nn.Dropout(drop_p)
            )),
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                FeedForwardBlock(emb_size, expansion=forward_expansion, drop_p=forward_drop_p),
                nn.Dropout(drop_p)
            ))
        )

class TransformerEncoder(nn.Sequential):
    def __init__(self, depth, emb_size):
        super().__init__(*[TransformerEncoderBlock(emb_size) for _ in range(depth)])

class ClassificationHead(nn.Sequential):
    def __init__(self, emb_size, n_classes):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(2440, 256),
            nn.ELU(),
            nn.Dropout(0.5),
            nn.Linear(256, 32),
            nn.ELU(),
            nn.Dropout(0.3),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        x = x.contiguous().view(x.size(0), -1)
        out = self.fc(x)
        return x, out

class Conformer(nn.Sequential):
    def __init__(self, emb_size=40, depth=6, n_classes=4, **kwargs):
        super().__init__(
            PatchEmbedding(emb_size),
            TransformerEncoder(depth, emb_size),
            ClassificationHead(emb_size, n_classes)
        )

# =========================================== 5. FINAL FIXED: PhysioNet eegmmidb Data Loading
def load_eegmmidb_subject(sub_id, base_path):
    """
    Load eegmmidb data for a single subject (100% bug-free channel handling)
    Task mapping: T1=Left Hand(0), T2=Right Hand(1), T3=Foot(2), T4=Tongue(3)
    """
    # Basic configuration
    sfreq = 160  # Original sampling rate of eegmmidb (160Hz)
    target_sfreq = 250  # Target sampling rate for model input
    t_start = 0.5  # Start time for trial extraction (0.5s after stimulus)
    t_end = 4.5  # End time for trial extraction (4.5s after stimulus)

    # Run numbers corresponding to motor imagery tasks
    runs = [11, 12, 13, 14]
    sub_str = f'S{sub_id:03d}'  # Subject folder format: S001, S002...
    sub_path = os.path.join(base_path, sub_str)  # Full path to subject data

    all_trials = []  # Store all EEG trial data
    all_labels = []  # Store all trial labels

    for run in runs:
        # Build EDF file path for current run
        edf_file = os.path.join(sub_path, f'{sub_str}R{run:02d}.edf')
        if not os.path.exists(edf_file):
            # Skip if file not found (error tolerance)
            print(f" Warning: File {edf_file} not found, skipping run {run}")
            continue

        # Read EDF file: preload loads data into RAM, verbose disables logs
        raw = mne.io.read_raw_edf(edf_file, preload=True, verbose=False)

        # Use MNE built-in pick_types
        raw.pick_types(eeg=True, exclude=[])
        # Force keep the first 64 EEG channels to avoid inconsistent channel counts
        if len(raw.ch_names) > 64:
            raw.pick(raw.ch_names[:64], verbose=False)

        # Resample to target sampling rate
        raw.resample(target_sfreq, verbose=False)

        # Extract events (stimulus markers) from annotations
        events, event_id = mne.events_from_annotations(raw, verbose=False)

        # Task mapping
        task_map = {
            11: {'T1': 0, 'T2': 1},
            12: {'T1': 0, 'T2': 3},
            13: {'T1': 2, 'T2': 3},
            14: {'T1': 2, 'T2': 1},
        }

        # Split trials
        data = raw.get_data() * 1e6
        sfreq_new = raw.info['sfreq']
        start_idx = int(t_start * sfreq_new)
        end_idx = int(t_end * sfreq_new)
        n_samples = end_idx - start_idx

        # Iterate over all events to extract trials
        for event in events:
            onset = event[0] # Start sample of the event
            trigger = event[2] # Trigger ID
            # Get trigger string (T1/T2) from event ID
            trigger_str = list(event_id.keys())[list(event_id.values()).index(trigger)]

            # Keep only T1/T2 (motor imagery tasks)
            if trigger_str not in ['T1', 'T2']:
                continue

            # Get label
            if run not in task_map or trigger_str not in task_map[run]:
                continue
            label = task_map[run][trigger_str]

            # Extract data
            trial_start = onset + start_idx
            trial_end = onset + end_idx
            if trial_end > data.shape[1]:
                continue # Skip if trial exceeds data length

            trial_data = data[:, trial_start:trial_end]

            # Ensure length
            if trial_data.shape[1] < n_samples:
                pad_len = n_samples - trial_data.shape[1]
                trial_data = np.pad(trial_data, ((0, 0), (0, pad_len)), mode='constant')
            else:
                trial_data = trial_data[:, :n_samples]

            # Ensure 64 channels
            if trial_data.shape[0] > 64:
                trial_data = trial_data[:64, :]
            elif trial_data.shape[0] < 64:
                pad_len = 64 - trial_data.shape[0]
                trial_data = np.pad(trial_data, ((0, pad_len), (0, 0)), mode='constant')

            # Append to trial list
            all_trials.append(trial_data)
            all_labels.append(label)

    # Return None if no valid trials
    if len(all_trials) == 0:
        print(f"Error: No valid trials for subject {sub_id}, skipping")
        return None, None

    # Convert to model input shape (n_trials, 1, 64, 1000)
    data = np.stack(all_trials, axis=0)
    data = np.expand_dims(data, axis=1)  # Add channel dimension
    labels = np.array(all_labels)

    print(f"Loaded subject {sub_id}: {len(all_trials)} trials, {all_trials[0].shape[0]} channels")
    return data, labels

# ======================= 6. FINAL FIXED: Training Pipeline Class ========================
class Exp():
    def __init__(self, nsub):
        # Initialize parent class
        super(Exp, self).__init__()
        # Load hyperparameters from global configuration
        self.batch_size = batch_size
        self.n_epochs = n_epochs
        self.c_dim = 4
        self.lr = learning_rate
        self.bl = 0.5
        self.b2 = 0.999
        self.nSub = nsub
        self.root = data_path

        self.log_write = open(f"{result_path}/log_subject{self.nSub}.txt", "w")
        # Define loss function (CrossEntropyLoss for classification) and move to GPU
        self.criterion_cls = torch.nn.CrossEntropyLoss().to(device)
        # Initialize Conformer model and move to GPU
        self.model = Conformer().to(device)
        # DataParallel optional, remove if single GPU
        # self.model = nn.DataParallel(self.model)

    def interaug(self, timg, label):
        aug_data = []  # Store augmented EEG data
        aug_label = []  # Store augmented labels
        batch_per_cls = int(self.batch_size / 4)  # Number of augmented samples per class

        for cls4aug in range(1, 5):  # Labels are 1,2,3,4
            # Get indices of samples for current class
            cls_idx = np.where(label == cls4aug)[0]
            if len(cls_idx) == 0:
                continue
            tmp_data = timg[cls_idx] # Get EEG data for current class
            tmp_label = label[cls_idx] # Get labels for current class

            # Generate augmented data Time-wise segment replacement
            tmp_aug_data = np.zeros((batch_per_cls, 1, 64, 1000))
            for ri in range(batch_per_cls):
                for rj in range(8):
                    # Fix random index to avoid out-of-bounds
                    rand_idx = np.random.randint(0, tmp_data.shape[0])
                    tmp_aug_data[ri, :, :, rj*125:(rj+1)*125] = tmp_data[rand_idx, :, :, rj*125:(rj+1)*125]

            # Fix label length, automatically repeat
            if len(tmp_label) < batch_per_cls:
                repeat_times = int(np.ceil(batch_per_cls / len(tmp_label)))
                tmp_label = np.tile(tmp_label, repeat_times)[:batch_per_cls]

            aug_data.append(tmp_aug_data)
            aug_label.append(tmp_label[:batch_per_cls])

        if len(aug_data) == 0:
            return None, None

        # Concatenate arrays, completely solve dimension issues
        aug_data = np.concatenate(aug_data, axis=0)
        aug_label = np.concatenate(aug_label, axis=0)

        # Shuffle
        aug_shuffle = np.random.permutation(len(aug_data))
        aug_data = aug_data[aug_shuffle]
        aug_label = aug_label[aug_shuffle]

        # Convert to tensor
        aug_data = torch.from_numpy(aug_data).to(device).float()
        aug_label = torch.from_numpy(aug_label - 1).to(device).long()  # Convert to 0~3

        return aug_data, aug_label

    # Load EEG data and labels for current subject
    def get_source_data(self):
        self.allData, self.allLabel = load_eegmmidb_subject(self.nSub, self.root)
        if self.allData is None:
            return None, None, None, None

        n_trials = len(self.allData)
        n_train = int(0.8 * n_trials)

        # Shuffle
        shuffle_num = np.random.permutation(n_trials)
        self.allData = self.allData[shuffle_num]
        self.allLabel = self.allLabel[shuffle_num]

        # Split
        train_data = self.allData[:n_train]
        train_label = self.allLabel[:n_train]
        test_data = self.allData[n_train:]
        test_label = self.allLabel[n_train:]

        # Standardization (using training data stats)
        target_mean = np.mean(train_data)
        target_std = np.std(train_data)
        self.allData = (self.allData - target_mean) / target_std
        train_data = (train_data - target_mean) / target_std
        test_data = (test_data - target_mean) / target_std

        return train_data, train_label+1, test_data, test_label+1

    def train(self):
        img, label, test_data, test_label = self.get_source_data()
        if img is None:
            return 0, 0, 0, 0

        # Convert train data to tensor and create DataLoader
        img = torch.from_numpy(img)
        label = torch.from_numpy(label - 1)  # Make label 0~3
        dataset = TensorDataset(img, label)
        self.dataLoader = DataLoader(dataset=dataset, batch_size=self.batch_size, shuffle=True)

        test_data = torch.from_numpy(test_data)
        test_label = torch.from_numpy(test_label - 1)
        test_dataset = TensorDataset(test_data, test_label)
        self.test_dataLoader = DataLoader(dataset=test_dataset, batch_size=self.batch_size)

        # Initialize Adam optimizer with model parameters and hyperparameters
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr, betas=(self.bl, self.b2))

        bestAcc = 0
        averAcc = 0
        num = 0
        Y_true = None
        Y_pred = None

        print(f'==== Start Training Subject {self.nSub} | Sample Count: {len(img)} ====')
        for e in range(self.n_epochs):
            self.model.train()
            for i, (img_batch, label_batch) in enumerate(self.dataLoader):
                img_batch = img_batch.to(device).float()
                label_batch = label_batch.to(device).long()

                # Data augmentation
                aug_data, aug_label = self.interaug(self.allData, self.allLabel+1)
                if aug_data is not None:
                    img_batch = torch.cat((img_batch, aug_data))
                    label_batch = torch.cat((label_batch, aug_label))

                _, outputs = self.model(img_batch)
                loss = self.criterion_cls(outputs, label_batch)
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                # Compute training accuracy for current batch
                _, train_pred = torch.max(outputs, 1)
                train_correct = (train_pred == label_batch).sum().item()
                train_acc = train_correct / label_batch.size(0)

            # ==================== Test Phase ====================
            self.model.eval()
            test_loss = 0.0
            correct = 0
            total = 0
            all_preds = []
            all_labels = []

            with torch.no_grad():
                for test_batch, test_batch_label in self.test_dataLoader:
                    test_batch = test_batch.to(device).float()
                    test_batch_label = test_batch_label.to(device).long()

                    _, outputs_test = self.model(test_batch)
                    loss_t = self.criterion_cls(outputs_test, test_batch_label)

                    test_loss += loss_t.item()
                    _, predicted = torch.max(outputs_test, 1)

                    total += test_batch_label.size(0)
                    correct += (predicted == test_batch_label).sum().item()

                    all_preds.append(predicted)
                    all_labels.append(test_batch_label)

            test_loss = test_loss / len(self.test_dataLoader)
            acc = correct / total
            Y_pred = torch.cat(all_preds)
            Y_true = torch.cat(all_labels)

            if (e+1) % 20 == 0:
                print(f'Epoch: {e+1:4d} | Train loss: {loss.item():.4f} | Train acc: {train_acc:.4f} | Test acc: {acc:.4f}')
                self.log_write.write(f"{e+1} {acc}\n")

            num += 1
            averAcc += acc
            if acc > bestAcc:
                bestAcc = acc
                best_Y_true = Y_true
                best_Y_pred = Y_pred

        torch.save(self.model.state_dict(), f'{result_path}/model_sub{self.nSub}.pth')
        averAcc = averAcc / num
        print(f'==== Subject {self.nSub} Training Completed | Best Accuracy: {bestAcc:.4f} ====\n')
        print(f'==== Subject {self.nSub} Average Epoch Accuracy: {averAcc:.4f} ====\n')
        self.log_write.write('The average accuracy is: ' + str(averAcc) + "\n")
        self.log_write.write('The best accuracy is: ' + str(bestAcc) + "\n")
        self.log_write.close()
        return bestAcc, averAcc, best_Y_true, best_Y_pred

# ========================= Main Function ============================
def main():
    best_total = 0
    aver_total = 0
    valid_count = 0
    result_write = open(f"{result_path}/all_subjects_result.txt", "w")
    print(f'===== EEG Conformer @ PhysioNet eegmmidb Training Start =====\n')

    for i in range(1, n_subjects+1):
        starttime = datetime.datetime.now()
        sub_id = i
        exp = Exp(sub_id)
        bestAcc, averAcc, Y_true, Y_pred = exp.train()

        if bestAcc > 0:
            result_write.write(f'Subject {sub_id} | Best accuracy: {bestAcc:.6f}\n')
            result_write.write(f'Subject {sub_id} | Average accuracy: {averAcc:.6f}\n')
            best_total += bestAcc
            aver_total += averAcc
            valid_count += 1

            if valid_count == 1:
                yt_all = Y_true
                yp_all = Y_pred
            elif Y_true is not None:
                yt_all = torch.cat((yt_all, Y_true))
                yp_all = torch.cat((yp_all, Y_pred))

        endtime = datetime.datetime.now()

    if valid_count > 0:
        best_total = best_total / valid_count
        aver_total = aver_total / valid_count
        print(f"===== All Training Completed | Valid Subjects: {valid_count} =====")
        print(f"Average Best Accuracy: {best_total:.6f}")
        print(f"Average Epoch Accuracy: {aver_total:.6f}")
        result_write.write(f"*Average Best Accuracy: {best_total:.6f}\n")
        result_write.write(f"*Average Epoch Accuracy: {aver_total:.6f}\n")
        result_write.close()

    print(f"\nAll logs and model weights saved to: {result_path}")

if __name__ == "__main__":
    main()

Checking/downloading PhysioNet eegmmidb dataset...
Dataset already exists, skipping download.
Using device: cuda
===== EEG Conformer @ PhysioNet eegmmidb Training Start =====

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Loaded subject 1: 60 trials, 64 channels
==== Start Training Subject 1 | Sample Count: 48 ====
Epoch:   20 | Train loss: 0.5421 | Train acc: 0.7708 | Test acc: 0.5000
Epoch:   40 | Train loss: 0.1930 | Train acc: 0.9375 | Test acc: 0.7500
Epoch:   60 | Train loss: 0.0670 | Train acc: 0.9583 | Test acc: 0.7500
Epoch:   80 | Train loss: 0.0930 | Train acc: 0.9792 | Test acc: 0.6667
Epoch:  100 | Train loss: 0.0379 | Train acc: 1.0000 | Test acc: 0.8333
Epoch:  120 | Train loss: 0.0357 | Train acc: 1.0000 | Test a